<a href="https://colab.research.google.com/github/alearecuest/anyoneai-exercises-sprint_2/blob/main/15_1_2_PRACTICE_MLP_for_CIFAR_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cifar-10 classification

Cifar-10 is a dataset having 60000 color images with 32x32x3 pixels each one.

Each image belongs to one out of 10 possible categories. Having exactly 6000 images per class.

We have 50000 images for training and 10000 for testing.

Let's use MLP models to solve this!

In [1]:
from keras.datasets import cifar10

## Load data

Keras will download it automatically.

In [2]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print('x_train shape: {}'.format(x_train.shape))
print('x_test shape: {}'.format(x_test.shape))

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
x_train shape: (50000, 32, 32, 3)
x_test shape: (10000, 32, 32, 3)


## [Complete] Plot some images for each class

In [3]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler


x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_test_flat = x_test.reshape((x_test.shape[0], -1))


# Initialize the scalers
min_max_scaler = MinMaxScaler()
standard_scaler = StandardScaler()

# Apply MinMaxScaler
x_train_minmax = min_max_scaler.fit_transform(x_train_flat)
x_test_minmax = min_max_scaler.transform(x_test_flat)

# Apply StandardScaler
x_train_standard = standard_scaler.fit_transform(x_train_flat)
x_test_standard = standard_scaler.transform(x_test_flat)

## Feature engineering

- Convert your data shape so you can use them with your MLP
- Scale your images
  - Try using MinMax and StandardScaler, compare both methods

In [4]:
import numpy as np

# Example data
x_train = np.random.rand(50000, 32, 32, 3)
x_test = np.random.rand(10000, 32, 32, 3)

# Reshape the data
x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_test_flat = x_test.reshape((x_test.shape[0], -1))

print(f"Reshaped x_train: {x_train_flat.shape}")
print(f"Reshaped x_test: {x_test_flat.shape}")

Reshaped x_train: (50000, 3072)
Reshaped x_test: (10000, 3072)


In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.datasets import cifar10

# Load data
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

# Flatten images for MLP
x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_test_flat = x_test.reshape((x_test.shape[0], -1))
print("Flattened shapes:", x_train_flat.shape, x_test_flat.shape)

# Scale data: MinMax and Standard
minmax = MinMaxScaler()
standard = StandardScaler()

x_train_minmax = minmax.fit_transform(x_train_flat)
x_test_minmax = minmax.transform(x_test_flat)

x_train_standard = standard.fit_transform(x_train_flat)
x_test_standard = standard.transform(x_test_flat)

# Model builder
def create_mlp_model(input_shape, layers_config, activation, weight_init, dropout_rate, l2_reg):
    model = Sequential()
    # first Dense with input shape
    for i, units in enumerate(layers_config):
        if i == 0:
            model.add(Dense(units, activation=activation, kernel_initializer=weight_init,
                            kernel_regularizer=l2(l2_reg), input_shape=input_shape))
        else:
            model.add(Dense(units, activation=activation, kernel_initializer=weight_init,
                            kernel_regularizer=l2(l2_reg)))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    model.add(Dense(10, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Training and evaluation (explicit args)
def train_and_evaluate(x_train, y_train, x_test, y_test,
                       layers_config, activation, weight_init, dropout_rate, l2_reg,
                       epochs=10, batch_size=64, verbose=2):
    input_shape = (x_train.shape[1],)
    model = create_mlp_model(input_shape, layers_config, activation, weight_init, dropout_rate, l2_reg)
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)
    model.fit(x_train, y_train, epochs=epochs, batch_size=batch_size,
              validation_data=(x_test, y_test), callbacks=[es], verbose=verbose)
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    # Get predictions for classification report
    y_pred = model.predict(x_test, batch_size=batch_size, verbose=0)
    y_pred_labels = np.argmax(y_pred, axis=1)
    return {
        "model": model,
        "test_acc": test_acc,
        "y_pred_labels": y_pred_labels
    }

# Configs - note the key name 'layers' (we will unpack manually below)
configs = [
    {'layers': [128, 64], 'activation': 'relu', 'weight_init': 'he_normal', 'dropout_rate': 0.5, 'l2_reg': 1e-4},
    {'layers': [256, 128, 64], 'activation': 'tanh', 'weight_init': 'glorot_uniform', 'dropout_rate': 0.3, 'l2_reg': 1e-3},
    {'layers': [512, 256], 'activation': 'relu', 'weight_init': 'he_uniform', 'dropout_rate': 0.4, 'l2_reg': 1e-2},
    {'layers': [128, 128, 128], 'activation': 'sigmoid', 'weight_init': 'glorot_normal', 'dropout_rate': 0.2, 'l2_reg': 1e-5}
]

# Evaluate configs (be careful: training multiple large models is slow)
results = []
for i, cfg in enumerate(configs, 1):
    print(f"\n=== Config {i}/{len(configs)}: {cfg} on MinMax scaler ===")
    res_min = train_and_evaluate(x_train_minmax, y_train, x_test_minmax, y_test,
                                 cfg['layers'], cfg['activation'], cfg['weight_init'],
                                 cfg['dropout_rate'], cfg['l2_reg'],
                                 epochs=10, batch_size=128, verbose=2)
    print(f"MinMax test accuracy: {res_min['test_acc']:.4f}")
    print(f"\n=== Config {i}/{len(configs)}: {cfg} on Standard scaler ===")
    res_std = train_and_evaluate(x_train_standard, y_train, x_test_standard, y_test,
                                 cfg['layers'], cfg['activation'], cfg['weight_init'],
                                 cfg['dropout_rate'], cfg['l2_reg'],
                                 epochs=10, batch_size=128, verbose=2)
    print(f"Standard test accuracy: {res_std['test_acc']:.4f}")
    results.append({'config': cfg, 'minmax': res_min, 'standard': res_std})

# Print a summary table of accuracies
for r in results:
    print("Config:", r['config'])
    print("  MinMax acc:", r['minmax']['test_acc'])
    print("  Standard acc:", r['standard']['test_acc'])
    print("--------------------------------------------------")

# Choose best model on MinMax for detailed report (example)
best = max(results, key=lambda rr: rr['minmax']['test_acc'])
best_model = best['minmax']['model']
y_pred_best = best['minmax']['y_pred_labels']

print("\nBest config on MinMax:", best['config'])
print("Test accuracy:", best['minmax']['test_acc'])

# Classification report and confusion matrix (on test set)
print("\nClassification report (best MinMax model):")
print(classification_report(y_test, y_pred_best, digits=4))

print("\nConfusion matrix (best MinMax model):")
cm = confusion_matrix(y_test, y_pred_best)
print(cm)

x_train shape: (50000, 32, 32, 3)
x_test shape: (10000, 32, 32, 3)
Flattened shapes: (50000, 3072) (10000, 3072)

=== Config 1/4: {'layers': [128, 64], 'activation': 'relu', 'weight_init': 'he_normal', 'dropout_rate': 0.5, 'l2_reg': 0.0001} on MinMax scaler ===


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
391/391 - 4s - 11ms/step - accuracy: 0.1133 - loss: 2.3219 - val_accuracy: 0.1601 - val_loss: 2.2480
Epoch 2/10
391/391 - 4s - 9ms/step - accuracy: 0.1297 - loss: 2.2746 - val_accuracy: 0.1770 - val_loss: 2.2116
Epoch 3/10
391/391 - 5s - 12ms/step - accuracy: 0.1340 - loss: 2.2569 - val_accuracy: 0.1535 - val_loss: 2.2390
Epoch 4/10
391/391 - 4s - 9ms/step - accuracy: 0.1368 - loss: 2.2508 - val_accuracy: 0.1809 - val_loss: 2.1718
Epoch 5/10
391/391 - 4s - 10ms/step - accuracy: 0.1362 - loss: 2.2476 - val_accuracy: 0.1871 - val_loss: 2.1727
Epoch 6/10
391/391 - 4s - 9ms/step - accuracy: 0.1391 - loss: 2.2410 - val_accuracy: 0.1824 - val_loss: 2.1656
Epoch 7/10
391/391 - 3s - 9ms/step - accuracy: 0.1406 - loss: 2.2390 - val_accuracy: 0.1826 - val_loss: 2.1826
Epoch 8/10
391/391 - 4s - 9ms/step - accuracy: 0.1415 - loss: 2.2371 - val_accuracy: 0.1752 - val_loss: 2.1786
Epoch 9/10
391/391 - 3s - 9ms/step - accuracy: 0.1386 - loss: 2.2353 - val_accuracy: 0.1878 - val_loss: 2.154

## Model training

Use keras tuner to find the best hyperparameters


In [ ]:
# Model training cell
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tensorflow.keras.datasets import cifar10

# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()

# Flatten images for MLP
x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_test_flat = x_test.reshape((x_test.shape[0], -1))

# Scale data: MinMax and Standard
minmax = MinMaxScaler()
standard = StandardScaler()
x_train_minmax = minmax.fit_transform(x_train_flat)
x_test_minmax = minmax.transform(x_test_flat)
x_train_standard = standard.fit_transform(x_train_flat)
x_test_standard = standard.transform(x_test_flat)

# Model builder
def create_mlp_model(input_shape, layers_config, activation, weight_init, dropout_rate, l2_reg):
    model = Sequential()
    for i, units in enumerate(layers_config):
        if i == 0:
            model.add(Dense(units, activation=activation, kernel_initializer=weight_init,
                            kernel_regularizer=l2(l2_reg), input_shape=input_shape))
        else:
            model.add(Dense(units, activation=activation, kernel_initializer=weight_init,
                            kernel_regularizer=l2(l2_reg)))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    model.add(Dense(10, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def train_and_evaluate(x_train, y_train, x_test, y_test,
                       layers_config, activation, weight_init, dropout_rate, l2_reg,
                       epochs=10, batch_size=128, verbose=2):
    input_shape = (x_train.shape[1],)
    model = create_mlp_model(input_shape, layers_config, activation, weight_init, dropout_rate, l2_reg)
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)
    model.fit(x_train, y_train, epochs=epochs, batch_size=batch_size,
              validation_data=(x_test, y_test), callbacks=[es], verbose=verbose)
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    y_pred = model.predict(x_test, batch_size=batch_size, verbose=0)
    y_pred_labels = np.argmax(y_pred, axis=1)
    return {"model": model, "test_acc": test_acc, "y_pred_labels": y_pred_labels}

configs = [
    {'layers': [128, 64], 'activation': 'relu', 'weight_init': 'he_normal', 'dropout_rate': 0.5, 'l2_reg': 1e-4},
    {'layers': [256, 128, 64], 'activation': 'tanh', 'weight_init': 'glorot_uniform', 'dropout_rate': 0.3, 'l2_reg': 1e-3},
    {'layers': [512, 256], 'activation': 'relu', 'weight_init': 'he_uniform', 'dropout_rate': 0.4, 'l2_reg': 1e-2},
    {'layers': [128, 128, 128], 'activation': 'sigmoid', 'weight_init': 'glorot_normal', 'dropout_rate': 0.2, 'l2_reg': 1e-5}
]

results = []
for i, cfg in enumerate(configs, 1):
    print(f"\n>>> Training config {i}/{len(configs)} on MinMax scaler: {cfg}")
    res_min = train_and_evaluate(x_train_minmax, y_train, x_test_minmax, y_test,
                                 cfg['layers'], cfg['activation'], cfg['weight_init'],
                                 cfg['dropout_rate'], cfg['l2_reg'],
                                 epochs=10, batch_size=128, verbose=2)
    print(f"MinMax test acc: {res_min['test_acc']:.4f}")

    print(f"\n>>> Training config {i}/{len(configs)} on Standard scaler: {cfg}")
    res_std = train_and_evaluate(x_train_standard, y_train, x_test_standard, y_test,
                                 cfg['layers'], cfg['activation'], cfg['weight_init'],
                                 cfg['dropout_rate'], cfg['l2_reg'],
                                 epochs=10, batch_size=128, verbose=2)
    print(f"Standard test acc: {res_std['test_acc']:.4f}")

    results.append({'config': cfg, 'minmax': res_min, 'standard': res_std})

best_min = max(results, key=lambda r: r['minmax']['test_acc'])
best_config_min = best_min['config']
best_model_min = best_min['minmax']['model']
best_pred_min = best_min['minmax']['y_pred_labels']
best_acc_min = best_min['minmax']['test_acc']

print("\nBest config on MinMax:", best_config_min)
print("Best MinMax test accuracy:", best_acc_min)


>>> Training config 1/4 on MinMax scaler: {'layers': [128, 64], 'activation': 'relu', 'weight_init': 'he_normal', 'dropout_rate': 0.5, 'l2_reg': 0.0001}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
391/391 - 5s - 12ms/step - accuracy: 0.1238 - loss: 2.3042 - val_accuracy: 0.1727 - val_loss: 2.2130
Epoch 2/10
391/391 - 3s - 9ms/step - accuracy: 0.1338 - loss: 2.2579 - val_accuracy: 0.1670 - val_loss: 2.1784
Epoch 3/10
391/391 - 5s - 14ms/step - accuracy: 0.1381 - loss: 2.2434 - val_accuracy: 0.1727 - val_loss: 2.1937
Epoch 4/10
391/391 - 3s - 9ms/step - accuracy: 0.1368 - loss: 2.2355 - val_accuracy: 0.1853 - val_loss: 2.1623
Epoch 5/10
391/391 - 4s - 11ms/step - accuracy: 0.1394 - loss: 2.2282 - val_accuracy: 0.1683 - val_loss: 2.1883
Epoch 6/10
391/391 - 3s - 8ms/step - accuracy: 0.1401 - loss: 2.2269 - val_accuracy: 0.1923 - val_loss: 2.1382
Epoch 7/10
391/391 - 4s - 9ms/step - accuracy: 0.1392 - loss: 2.2228 - val_accuracy: 0.1807 - val_loss: 2.1674
Epoch 8/10
391/391 - 4s - 10ms/step - accuracy: 0.1406 - loss: 2.2224 - val_accuracy: 0.1640 - val_loss: 2.1888
Epoch 9/10
391/391 - 4s - 10ms/step - accuracy: 0.1399 - loss: 2.2232 - val_accuracy: 0.1593 - val_loss: 2.1

## Model evaluation

Use different evaluation metrics and your testing dataset to report how good your model is.

Think using scikit-learn `classification_report` and confussion matrix to generate good quality reports.


In [ ]:
# Model evaluation cell
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Best MinMax model test accuracy (from training cell):", best_acc_min)

# Classification report
print("\nClassification report (best MinMax model):")
print(classification_report(y_test, best_pred_min, digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, best_pred_min)
print("\nConfusion matrix:")
print(cm)

# Heatmap visualization
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (best MinMax model)")
plt.show()